In [0]:
SHOW TABLES IN bronze_olist;

In [0]:
USE SCHEMA bronze_olist;

In [0]:
CREATE SCHEMA IF NOT EXISTS silver_olist;

##### 1. Table Orders

In [0]:
--View table Orders
SELECT * FROM orders;

In [0]:
--Check if there is null value for order_purchase_timestamp
SELECT *
FROM orders
WHERE order_purchase_timestamp IS NULL


In [0]:
-- Check whether each order is unique
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id) AS distinct_rows,
  (COUNT(*) = COUNT(DISTINCT order_id)) AS is_unique
FROM orders;


In [0]:
--Change column types & extract dates, then load to silver layer

CREATE TABLE IF NOT EXISTS silver_olist.orders AS 
  SELECT 
    order_id,
    customer_id,
    order_status,
    CAST(order_purchase_timestamp AS TIMESTAMP) as order_purchase_ts,
    TO_DATE(order_purchase_timestamp) as order_purchase_date,
    CAST(order_approved_at AS TIMESTAMP) as order_approved_ts,
    TO_DATE(order_approved_at) as order_approved_date,
    CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_ts,
    TO_DATE(order_delivered_carrier_date) as order_delivered_carrier_date,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_ts,
    TO_DATE(order_delivered_customer_date) as order_delivered_customer_date,
    TO_DATE(order_estimated_delivery_date) as order_estimated_delivery_date
  FROM orders;

#####2.Table customers & geolocation

In [0]:
--View customers table
SELECT * FROM customers LIMIT 5

In [0]:
-- View geolocation table
SELECT * FROM geolocation LIMIT 5

In [0]:
CREATE TABLE IF NOT EXISTS silver_olist.customers AS
  SELECT 
    c.customer_id, 
    c.customer_unique_id,
    c.customer_zip_code_prefix as zip_code_prefix, 
    INITCAP(TRIM(c.customer_city)) AS city, 
    c.customer_state as state,
    g.geolocation_lat as latitude,
    g.geolocation_lng as longitude 
  FROM customers c
  LEFT JOIN geolocation g
  ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix

#####3. Table order_items

In [0]:
SELECT * FROM order_items

In [0]:
CREATE TABLE IF NOT EXISTS silver_olist.order_items AS
  SELECT 
    order_id,
    product_id, 
    order_item_id,
    seller_id,
    CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_ts,
    TO_DATE(shipping_limit_date) AS shipping_limit_date,
    CAST(price AS DOUBLE) AS price,
    CAST(freight_value AS DOUBLE) AS freight_value
  FROM order_items


#####4. Table order_payments

In [0]:
SELECT * FROM order_payments

In [0]:

--Cast values to right type
--Put to silver layer
CREATE TABLE IF NOT EXISTS silver_olist.order_payments AS
  SELECT 
    order_id,
    CAST(payment_sequential AS INT) as payment_sequential,
    payment_type,
    CAST(payment_installments AS INT) AS payment_installments,
    CAST(payment_value AS DOUBLE) AS payment_value  FROM order_payments
  

#####5. Table order_reviews

In [0]:
SELECT * FROM order_reviews

In [0]:
-- Choose order_id, review_id, review_score to the silver layer table
CREATE TABLE IF NOT EXISTS silver_olist.order_reviews AS
  SELECT 
    order_id,
    review_id,
    CAST(review_score AS INT) as review_score
  FROM order_reviews

#####6. Join products and product_category_name and load to silver layer

In [0]:
--View products table
select * from products limit 5

In [0]:
--View product_category_name table
SELECT * FROM product_category_name LIMIT 5

In [0]:
--Left join products on product_category_name to get product_category_name_english column, modify type of some columns in products table. Then, load to products table in silver layer
CREATE TABLE IF NOT EXISTS silver_olist.products AS
  SELECT 
    p.product_id,
    p.product_category_name,
    INITCAP(REPLACE(TRIM(c.product_category_name_english),'_',' ')) AS product_category_name_english,
    CAST(p.product_name_lenght AS INT) AS product_name_lenght,
    CAST(p.product_description_lenght AS INT) AS product_description_lenght,
    CAST(p.product_photos_qty AS INT) AS product_photos_qty,
    CAST(p.product_weight_g AS INT) AS product_weight_g,
    CAST(p.product_length_cm AS INT) AS product_length_cm,
    CAST(p.product_height_cm AS INT) AS product_height_cm,
    CAST(p.product_width_cm AS INT) AS product_width_cm
  FROM products p
  LEFT JOIN product_category_name c
  ON p.product_category_name = c.product_category_name

#####7. Table sellers & geolocation

In [0]:
--View sellers table
SELECT * FROM sellers limit 5

In [0]:
-- View geolocation table;
SELECT * FROM geolocation 

In [0]:
--Left join sellers table with geolocation table to get longitude and latitude for each seller. 
--Load the result to silver layer
CREATE TABLE IF NOT EXISTS silver_olist.sellers AS
  SELECT 
    s.seller_id,
    s.seller_zip_code_prefix AS zip_code_prefix,
    INITCAP(s.seller_city) AS city,
    s.seller_state as state,
    g.geolocation_lat AS latitude,
    g.geolocation_lng AS longitude
  FROM sellers s
  LEFT JOIN geolocation g
  ON s.seller_zip_code_prefix = g.geolocation_zip_code_prefix